In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))  # points to src/
from shared_modeling import load_or_create_master_split_ids, run_model_experiment

In [13]:
predictors = ['oDM', 'acog_PEgHTN', 'ChronHTN']  # diabetes, preeclampsia, chronic hypertension
df_health = pd.read_csv('../../Data/PREGNANCY_OUTCOMES.csv', usecols=predictors + ['PublicID'])
df_outcomes = pd.read_csv('../../Data/Modified/Outcome.csv', usecols=['PublicID', 'MH_outcome'])

# Mental health source datasets used as additional predictors in the maternal health models.
v2i_cols = ['PublicID'] + [f'V2IA{i:02d}' for i in range(1, 26)]
v1e_cols = ['PublicID', 'V1EA01', 'V1EA02a', 'V1EA02b', 'V1EA02c', 'V1EA02d', 'V1EA02e', 'V1EA02f', 'V1EA02g', 'V1EA02h', 'V1EA02i', 'V1EA02j', 'V1EA02k', 'V1EA02l']
v3j_cols = ['PublicID'] + [f'V3JA01{letter}' for letter in 'abcdefghij'] + [f'V3JA02{letter}' for letter in 'abcdefghij']
v1a_cols = ['PublicID', 'V1AH04', 'V1AH05', 'V1AH07', 'V1AH08']
v3a_cols = ['PublicID', 'V3AG04', 'V3AG05', 'V3AG07', 'V3AG08']
v1c_cols = ['PublicID'] + [f'V1CA{i:02d}' for i in range(1, 11)]
v1h_cols = ['PublicID'] + [f'V1HA{i:02d}' for i in range(1, 21)]

df_v2i = pd.read_csv('../../Data/V2I.csv', usecols=v2i_cols)
df_v1e = pd.read_csv('../../Data/V1E.CSV', usecols=v1e_cols, encoding='ISO-8859-1')
df_v3j = pd.read_csv('../../Data/V3J.csv', usecols=v3j_cols)
df_v1a = pd.read_csv('../../Data/V1A.CSV', usecols=v1a_cols)
df_v3a = pd.read_csv('../../Data/V3A.CSV', usecols=v3a_cols)
df_v1c = pd.read_csv('../../Data/V1C.CSV', usecols=v1c_cols)
df_v1h = pd.read_csv('../../Data/V1H.CSV', usecols=v1h_cols)

# Create the master split once and persist it for reuse in other notebooks.
split_path = 'master_split_ids.csv'
train_ids, test_ids = load_or_create_master_split_ids(df_outcomes, split_path)


In [14]:
def compute_resilience_score(df):
    item_cols = [f'V2IA{i:02d}' for i in range(1, 26)]
    result = df[['PublicID']].copy()
    result['ResilienceTotalScore'] = df[item_cols].sum(axis=1)

    def resilience_level(score):
        if score <= 75:
            return 3
        elif score <= 100:
            return 2
        return 1

    result['ResilienceLevel'] = result['ResilienceTotalScore'].apply(resilience_level)
    return result


def compute_stress_average(df):
    relevant_columns = ['V1EA01', 'V1EA02a', 'V1EA02b', 'V1EA02c', 'V1EA02d', 'V1EA02e', 'V1EA02f', 'V1EA02g', 'V1EA02h', 'V1EA02i', 'V1EA02j', 'V1EA02k', 'V1EA02l']
    result = df[['PublicID']].copy()
    result['stress_average'] = df[relevant_columns].mean(axis=1)
    return result


def compute_hassles_uplifts(df):
    hassles_cols = [f'V3JA02{letter}' for letter in 'abcdefghij']
    uplifts_cols = [f'V3JA01{letter}' for letter in 'abcdefghij']
    result = df[['PublicID']].copy()
    result['FrequencyOfHassles'] = df[hassles_cols].gt(0).sum(axis=1)
    result['FrequencyOfUplifts'] = df[uplifts_cols].gt(0).sum(axis=1)
    result['IntensityOfHassles'] = np.where(
        result['FrequencyOfHassles'] > 0,
        df[hassles_cols].sum(axis=1) / result['FrequencyOfHassles'],
        0.0,
    )
    result['IntensityOfUplifts'] = np.where(
        result['FrequencyOfUplifts'] > 0,
        df[uplifts_cols].sum(axis=1) / result['FrequencyOfUplifts'],
        0.0,
    )
    result['HassleUpliftFrequencyRatio'] = np.where(
        result['FrequencyOfUplifts'] > 0,
        result['FrequencyOfHassles'] / result['FrequencyOfUplifts'],
        0.0,
    )
    result['HassleUpliftIntensityRatio'] = np.where(
        result['IntensityOfUplifts'] > 0,
        result['IntensityOfHassles'] / result['IntensityOfUplifts'],
        0.0,
    )
    return result


def compute_stress_level(df):
    reverse_columns = ['V1AH04', 'V1AH05', 'V1AH07', 'V1AH08', 'V3AG04', 'V3AG05', 'V3AG07', 'V3AG08']
    result = df[['PublicID']].copy()
    result['StressTotalScore'] = (6 - df[reverse_columns]).sum(axis=1)

    def stress_level(score):
        if 0 <= score <= 13:
            return 0
        elif 14 <= score <= 26:
            return 0.5
        elif 27 <= score <= 40:
            return 1
        return np.nan

    result['StressLevel'] = result['StressTotalScore'].apply(stress_level)
    return result


def compute_edinburgh_scores(df):
    result = df[['PublicID']].copy()
    total = pd.Series(0, index=df.index, dtype='float64')
    for i in range(1, 11):
        col = f'V1CA{i:02d}'
        if i == 10:
            total += (df[col] - 4).abs()
        elif i in {1, 2, 4}:
            total += df[col] - 1
        else:
            total += (df[col] - 4).abs()
    result['TotalEDINScore'] = total
    result['SubEDINScore'] = (result['TotalEDINScore'] >= 10).astype(int)
    return result


def compute_stai_scores(df):
    reverse = {1, 3, 6, 7, 10, 13, 14, 16, 19}
    result = df[['PublicID']].copy()
    total = pd.Series(0, index=df.index, dtype='float64')
    for i in range(1, 21):
        col = f'V1HA{i:02d}'
        if i in reverse and i > 9:
            total += (df[col] - 5).abs()
        else:
            total += df[col]
    result['TotalSTAIScore'] = total
    result['SubSTAIScore'] = (result['TotalSTAIScore'] >= 40).astype(int)
    return result


df_v2i_features = compute_resilience_score(df_v2i)
df_v1e_features = compute_stress_average(df_v1e)
df_v3j_features = compute_hassles_uplifts(df_v3j)
df_v1a_v3a_features = compute_stress_level(pd.merge(df_v1a, df_v3a, on='PublicID', how='inner'))
# df_v1c_features = compute_edinburgh_scores(df_v1c)
# df_v1h_features = compute_stai_scores(df_v1h)

mental_health_features = df_v2i_features.merge(df_v1e_features, on='PublicID', how='outer')
mental_health_features = mental_health_features.merge(df_v3j_features, on='PublicID', how='outer')
mental_health_features = mental_health_features.merge(df_v1a_v3a_features, on='PublicID', how='outer')
# mental_health_features = mental_health_features.merge(df_v1c_features, on='PublicID', how='outer')
# mental_health_features = mental_health_features.merge(df_v1h_features, on='PublicID', how='outer')


In [15]:
df = pd.merge(df_health, mental_health_features, on='PublicID', how='left')
df = pd.merge(df, df_outcomes, on='PublicID', how='inner')

feature_columns = [col for col in df.columns if col not in ['PublicID', 'MH_outcome']]
X = df[feature_columns]
y = df['MH_outcome']

train_df = df[df['PublicID'].isin(train_ids)].copy()
test_df = df[df['PublicID'].isin(test_ids)].copy()

X_train = train_df.drop(['MH_outcome', 'PublicID'], axis=1)
X_test = test_df.drop(['MH_outcome', 'PublicID'], axis=1)
y_train = train_df['MH_outcome']
y_test = test_df['MH_outcome']

y.value_counts()

MH_outcome
0    4591
1    3150
Name: count, dtype: int64

In [16]:
X.head()


,oDM,ChronHTN,acog_PEgHTN,ResilienceTotalScore,ResilienceLevel,stress_average,FrequencyOfHassles,FrequencyOfUplifts,IntensityOfHassles,IntensityOfUplifts,HassleUpliftFrequencyRatio,HassleUpliftIntensityRatio,StressTotalScore,StressLevel
0,3.0,2.0,7.0,102.0,1.0,1.846154,10.0,10.0,2.6,3.0,1.0,0.866667,24.0,0.5
1,3.0,2.0,7.0,106.0,1.0,1.538462,10.0,10.0,2.0,2.6,1.0,0.769231,18.0,0.5
2,3.0,2.0,7.0,107.0,1.0,2.076923,10.0,10.0,2.2,2.4,1.0,0.916667,17.0,0.5
3,3.0,2.0,7.0,120.0,1.0,1.692308,10.0,10.0,1.4,4.0,1.0,0.350000,10.0,0.0
4,3.0,2.0,7.0,97.0,2.0,2.000000,10.0,10.0,1.5,2.3,1.0,0.652174,18.0,0.5


In [17]:
# Run the LR pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'lr',
    feature_columns
)


Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found: {'classifier__C': 0.1, 'classifier__l1_ratio': 1.0}
Best Macro F1 Score: 0.7217
Model Coefficients:
num__oDM: 0.0
num__ChronHTN: -0.08462031908042408
num__acog_PEgHTN: -0.00013808903407582608
num__ResilienceTotalScore: -0.2527691291757995
num__ResilienceLevel: 0.12197358507836824
num__stress_average: 0.5023068916588417
num__FrequencyOfHassles: 0.0
num__FrequencyOfUplifts: 0.06361897983624072
num__IntensityOfHassles: 0.0
num__IntensityOfUplifts: 0.1787346956602595
num__HassleUpliftFrequencyRatio: 0.0
num__HassleUpliftIntensityRatio: 0.27325813994051584
num__StressTotalScore: 0.7063199555152916
num__StressLevel: -0.03333198491033657
Evaluation Metrics for LR with shared preprocessing and macro F1 grid search:
Accuracy: 0.7076
Precision: 0.6992
Recall: 0.7034
F1-score: 0.7005
ROC AUC: 0.7788


In [18]:
# Run the RF pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'rf',
    feature_columns
)


Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters found: {'classifier__max_depth': 18, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 700}
Best Macro F1 Score: 0.7207
Feature Importances:
num__oDM: 0.005005929610422907
num__ChronHTN: 0.004423952795349149
num__acog_PEgHTN: 0.025269873920738685
num__ResilienceTotalScore: 0.1478956376767254
num__ResilienceLevel: 0.04067845334664499
num__stress_average: 0.1817761763524296
num__FrequencyOfHassles: 9.646635599995892e-05
num__FrequencyOfUplifts: 0.00019452474810899702
num__IntensityOfHassles: 0.10445446915267272
num__IntensityOfUplifts: 0.07868486284584018
num__HassleUpliftFrequencyRatio: 0.00020907879320756391
num__HassleUpliftIntensityRatio: 0.11374349617889705
num__StressTotalScore: 0.2377921211280838
num__StressLevel: 0.05977495709487912
Evaluation Metrics for RF with shared preprocessing and macro F1 grid search:
Accuracy: 0.7159
Precision: 0.7057
Recall: 0.7

In [19]:
# Run the XGBoost pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'xgb',
    feature_columns
)


Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found: {'classifier__colsample_bytree': 0.8, 'classifier__learning_rate': 0.01, 'classifier__max_depth': 6, 'classifier__n_estimators': 100, 'classifier__subsample': 0.5}
Best Macro F1 Score: 0.7257
Feature Importances:
num__oDM: 0.028863895684480667
num__ChronHTN: 0.04145600274205208
num__acog_PEgHTN: 0.029757210984826088
num__ResilienceTotalScore: 0.07452241331338882
num__ResilienceLevel: 0.09360788017511368
num__stress_average: 0.12367536127567291
num__FrequencyOfHassles: 0.0
num__FrequencyOfUplifts: 0.0
num__IntensityOfHassles: 0.05036984011530876
num__IntensityOfUplifts: 0.029864247888326645
num__HassleUpliftFrequencyRatio: 0.0
num__HassleUpliftIntensityRatio: 0.03461916744709015
num__StressTotalScore: 0.2688651382923126
num__StressLevel: 0.2243988960981369
Evaluation Metrics for XGB with shared preprocessing and macro F1 grid search:
Accuracy: 0.7088
Precision: 0.6987
Recall: 0.7000
F1-score: 0.6993
R

In [20]:
# Run the SVM pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'svm',
    feature_columns
)


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters found: {'classifier__estimator__C': 1, 'classifier__estimator__kernel': 'linear'}
Best Macro F1 Score: 0.7216
Skipping feature-level SVM output to keep notebook output compact.
Evaluation Metrics for SVM with shared preprocessing and macro F1 grid search:
Accuracy: 0.7050
Precision: 0.6963
Recall: 0.7002
F1-score: 0.6976
ROC AUC: 0.7764
